# Day 08 — Pydantic and Databases
**Vidya Sankalp | Python for AI Engineering**

## What you'll learn
- Why data validation matters
- Create Pydantic models with `BaseModel`
- Add rules with `Field`
- Handle optional and restricted values
- Validate dictionaries and JSON
- Convert models back to dictionaries
- Understand how Python reads and writes database records
- Connect Pydantic validation to a PostgreSQL workflow

We'll use familiar **Customer, Product, Order, and Support Ticket** examples. Start at the top and run cells in order.

> **Setup:** Install Pydantic for the first sections with `pip install pydantic`. PostgreSQL sections are optional and require a running PostgreSQL database.

## 1. Why do we need validation?

Imagine an application receives a customer record from an API or an AI model. A plain dictionary can contain missing fields or incorrect values.

In [ ]:
customer_data = {
    "customer_id": "1001",
    "name": "Ravi Kumar",
    "email": "ravi@example.com"
}

print(customer_data["name"])
print("ID type:", type(customer_data["customer_id"]).__name__)

The ID is a string, even though our application may expect an integer. We also have no built-in rule that checks whether required fields exist.

**Pydantic** lets us describe the shape of the data and validate incoming values against it.

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

## 2. Create your first model

A Pydantic model is a class that inherits from `BaseModel`. The annotations describe the expected fields and types.

In [ ]:
class Customer(BaseModel):
    customer_id: int
    name: str
    email: str

customer = Customer(
    customer_id="1001",
    name="Ravi Kumar",
    email="ravi@example.com"
)

print(customer)
print("ID:", customer.customer_id)
print("ID type:", type(customer.customer_id).__name__)

Pydantic converts `"1001"` to the integer `1001` because the conversion is valid. It raises `ValidationError` when it cannot validate the input.

In [ ]:
try:
    invalid_customer = Customer(
        customer_id="not-a-number",
        name="Ravi Kumar",
        email="ravi@example.com"
    )
except ValidationError as error:
    print(error)

## 3. Add rules with `Field`

Types alone may not be enough. For example, a customer ID should be positive and a name should not be empty.

In [ ]:
class CustomerRecord(BaseModel):
    customer_id: int = Field(gt=0)
    name: str = Field(min_length=1, max_length=60)
    email: str = Field(min_length=5)
    city: str = "Hyderabad"

good_customer = CustomerRecord(
    customer_id=1001,
    name="Ravi Kumar",
    email="ravi@example.com"
)

print(good_customer)
print("City default:", good_customer.city)

In [ ]:
try:
    CustomerRecord(
        customer_id=-1,
        name="",
        email="bad"
    )
except ValidationError as error:
    for item in error.errors():
        print("Field:", item["loc"], "| Problem:", item["msg"])

**Common constraints**

| Constraint | Meaning |
|---|---|
| `gt=0` | Greater than zero |
| `ge=0` | Zero or greater |
| `min_length=1` | At least one character |
| `max_length=60` | No more than 60 characters |
| `default="Hyderabad"` | Use this value when the field is omitted |

For production email validation, use Pydantic's email validation support (`EmailStr`) and install its email extra. The simple `email` field above focuses on the core model concepts.

## 4. Optional fields and allowed values

Some fields may be absent. Other fields should accept only specific values. For example, an order status might be `Pending`, `Shipped`, or `Delivered`.

In [ ]:
class Order(BaseModel):
    order_id: int = Field(gt=0)
    customer_id: int = Field(gt=0)
    total_amount: float = Field(ge=0)
    status: Literal["Pending", "Shipped", "Delivered", "Cancelled"]
    notes: str | None = None

order = Order(
    order_id=5001,
    customer_id=1001,
    total_amount=2499.50,
    status="Shipped"
)

print(order)
print("Notes:", order.notes)

In [ ]:
try:
    Order(
        order_id=5002,
        customer_id=1001,
        total_amount=2499.50,
        status="On the way"  # not one of the allowed values
    )
except ValidationError as error:
    print(error)

`str | None = None` means the notes field may contain a string or `None`, and it is optional because it has a default.

`Literal[...]` restricts a field to the exact listed values.

## 5. Validate dictionaries and JSON

Incoming data often arrives as a Python dictionary or JSON text.

In [ ]:
# Validate a Python dictionary, such as an API or database record.
raw_order = {
    "order_id": "5003",
    "customer_id": "1001",
    "total_amount": "799.99",
    "status": "Pending"
}

order_from_dict = Order.model_validate(raw_order)
print(order_from_dict)
print(type(order_from_dict.total_amount).__name__)

In [ ]:
# Validate JSON text in one step.
order_json = '''
{
  "order_id": 5004,
  "customer_id": 1002,
  "total_amount": 1299.0,
  "status": "Delivered",
  "notes": "Leave at reception"
}
'''

order_from_json = Order.model_validate_json(order_json)
print(order_from_json)

## 6. Convert a model back to data

Use `model_dump()` when another part of your application needs a dictionary. Use `model_dump_json()` when you need JSON text.

In [ ]:
order_dict = order_from_json.model_dump()
order_json_output = order_from_json.model_dump_json()

print(type(order_dict).__name__, order_dict)
print(type(order_json_output).__name__, order_json_output)

## 7. A practical AI Engineering example: validate an AI classification

An AI model might classify a customer support message. Do not blindly trust its output: validate the category and required fields before using them.

In [ ]:
class SupportTicketResult(BaseModel):
    category: Literal["TRACK_ORDER", "CANCEL_ORDER", "REFUND", "GENERAL"]
    confidence: Literal["low", "medium", "high"]
    reason: str = Field(min_length=5)

ai_json = '''
{
  "category": "TRACK_ORDER",
  "confidence": "high",
  "reason": "The customer asks where their order is."
}
'''

ticket_result = SupportTicketResult.model_validate_json(ai_json)
print("Category:", ticket_result.category)
print("Confidence:", ticket_result.confidence)
print("Reason:", ticket_result.reason)

Try changing `category` to `"OTHER"` or removing `reason`. Run the cell again to see validation catch the problem.

**Important:** validation checks structure and rules; it does not prove that the AI's answer is factually correct.

## 8. Nested models: an employee with an address

Real records can contain nested objects and lists. Pydantic can validate these structures too.

In [ ]:
class Address(BaseModel):
    city: str
    state: str

class Employee(BaseModel):
    employee_id: int = Field(gt=0)
    name: str
    address: Address
    skills: list[str]

employee_data = {
    "employee_id": 101,
    "name": "Ananya",
    "address": {"city": "Hyderabad", "state": "TS"},
    "skills": ["Python", "SQL", "AWS"]
}

employee = Employee.model_validate(employee_data)
print(employee.name)
print(employee.address.city)
print(employee.skills)

## 9. Database basics: where Pydantic fits

A database stores records. Python sends SQL commands through a database driver.

- **Connection:** opens a session with the database.
- **Cursor:** executes SQL and reads results.
- **INSERT:** adds a row.
- **SELECT:** reads rows.
- **Commit:** saves changes made by a transaction.
- **Parameterized query:** passes values separately from SQL, helping prevent SQL injection.

The flow we want is:

`Database row → Python dictionary → Pydantic validation → application logic`

### Optional PostgreSQL setup

This section requires a PostgreSQL server and these packages:

```bash
pip install "psycopg[binary]" python-dotenv
```

Create a `.env` file (keep it private and never commit it to Git):

```text
DB_HOST=localhost
DB_PORT=5432
DB_NAME=shopsmart
DB_USER=postgres
DB_PASSWORD=your_password
```

The next cells do **not** connect automatically. Run them only after setting up your own database and credentials.

In [ ]:
# Optional: PostgreSQL connection helper.
import os
from dotenv import load_dotenv
import psycopg

load_dotenv()

def get_connection():
    required = ["DB_NAME", "DB_USER", "DB_PASSWORD"]
    missing = [key for key in required if not os.getenv(key)]
    if missing:
        raise ValueError(f"Missing database settings: {', '.join(missing)}")

    return psycopg.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=int(os.getenv("DB_PORT", "5432")),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

print("Connection helper defined. Connect only when PostgreSQL is configured.")

In [ ]:
# Optional: create a small customers table.
# Run this cell only when your PostgreSQL database is ready.
def create_customers_table():
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                CREATE TABLE IF NOT EXISTS customers (
                    customer_id INTEGER PRIMARY KEY,
                    name VARCHAR(60) NOT NULL,
                    email VARCHAR(120) NOT NULL,
                    city VARCHAR(60) NOT NULL
                )
            """)
    print("Customers table is ready.")

# Uncomment to run:
# create_customers_table()

In [ ]:
# Optional: insert a customer using parameters, not an f-string.
def insert_customer(customer_id, name, email, city):
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO customers (customer_id, name, email, city)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (customer_id) DO NOTHING
                """,
                (customer_id, name, email, city)
            )
    print("Insert checked for customer:", customer_id)

# Uncomment to run after creating the table:
# insert_customer(1001, "Ravi Kumar", "ravi@example.com", "Hyderabad")

In [ ]:
# Optional: fetch a row as a dictionary, then validate it.
def get_customer(customer_id):
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT customer_id, name, email, city FROM customers WHERE customer_id = %s",
                (customer_id,)
            )
            row = cur.fetchone()

    if row is None:
        return None

    return {
        "customer_id": row[0],
        "name": row[1],
        "email": row[2],
        "city": row[3]
    }

# Uncomment to query your database:
# row = get_customer(1001)
# if row is not None:
#     validated_customer = CustomerRecord.model_validate(row)
#     print(validated_customer)

## 10. Practice tasks

Try these before looking at the summary.

1. Create a `Product` model with `product_id`, `name`, `price`, and `stock`.
2. Require `price` to be greater than zero and `stock` to be zero or greater.
3. Add a `status` field to an `Order` that allows only `Pending`, `Shipped`, or `Delivered`.
4. Validate an order from a dictionary where numeric values arrive as strings.
5. Create a `SupportTicketResult` with an invalid category and inspect the error.
6. Create an `Employee` model with a nested `Address` and a list of skills.
7. Explain why SQL values should be passed as parameters rather than inserted into a query using an f-string.

## Day 08 recap

| Concept | Remember |
|---|---|
| `BaseModel` | Defines a validated data structure |
| Type annotations | Describe expected field types |
| `Field(...)` | Adds constraints and defaults |
| `Literal[...]` | Restricts values to an allowed set |
| `ValidationError` | Explains why input failed validation |
| `model_validate(dict)` | Validates a Python dictionary |
| `model_validate_json(text)` | Parses and validates JSON |
| `model_dump()` | Converts a model to a dictionary |
| SQL parameters | Keep data values separate from SQL text |
| Database + Pydantic | Validate records before application logic |

**Next:** Day 09 — HTTP requests and APIs.